In [0]:
%pip install yfinance

In [0]:
dbutils.library.restartPython()

In [0]:
%run "/Workspace/Users/eujin.lai.82@gmail.com/00_config"

In [0]:
import boto3
import pandas as pd
from io import StringIO
from botocore.client import Config

# ── MinIO Config ─────────────────────────────────────────
MINIO_ENDPOINT   = G_MINIO_ENDPOINT
MINIO_BUCKET     = "rawdatasets"
MINIO_KEY        = "cgs.csv"
 
MINIO_ACCESS_KEY = G_MINIO_ACCESS_KEY
MINIO_SECRET_KEY = G_MINIO_SECRET_KEY
# ─────────────────────────────────────────────────────────

# Create boto3 S3 client pointing to MinIO
s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,          # 🔑 Key difference from AWS S3
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),  # MinIO requires Sig v4
    region_name="us-east-1",              # Arbitrary but required by boto3
)

# Fetch CSV from MinIO
response = s3_client.get_object(Bucket=MINIO_BUCKET, Key=MINIO_KEY)
status   = response.get("ResponseMetadata", {}).get("HTTPStatusCode")

if status == 200:
    print(f"✅ Connected to MinIO. Status: {status}")
    csv_content = response["Body"].read().decode("utf-8")
    df = pd.read_csv(StringIO(csv_content))
    print(f"Loaded {len(df)} rows × {len(df.columns)} columns")
    display(df)
else:
    raise Exception(f"❌ Failed to fetch file. Status: {status}")

In [0]:
 
import pandas as pd
import yfinance as yf
import numpy as np

from io import StringIO
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType

# Decode bytes → string → wrap in StringIO for pandas
pdf = df

def get_info(symbol):
    try:
        info = yf.Ticker(symbol).info
        return pd.Series({
            "name": info.get("longName", "N/A"),
            "current_price": info.get("currentPrice", info.get("regularMarketPrice", None))
        })
    except:
        return pd.Series({"name": "N/A", "current_price": None})

# Apply row-wise — adds 2 new columns directly
pdf[["name", "current_price"]] = pdf["Symbol"].apply(get_info)



def clean_numeric_columns(df, columns):
    for col in columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            # Bare "-" or "--" → NaN
            .replace(r'^-+$', np.nan, regex=True)
            # Convert accounting negatives like (7.00) → -7.00
            .str.replace(r'^\((\d+\.?\d*)\)$', r'-\1', regex=True)
            # Remove mid-string minus (not a leading negative sign)
            .str.replace(r'(?!^)-', '', regex=True)
            # Strip all non-numeric characters except digits, dot, leading minus
            .str.replace(r'[^\d\.\-]', '', regex=True)
        )
        # ✅ Use pd.to_numeric directly — handles empty strings as NaN via errors='coerce'
        # No need for .replace('', np.nan) at all
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Specify which columns need cleaning
numeric_cols = [
    "LACP", "Qty.Hand", "Qty.Avai", "Qty.Q(S)",
    "Avg.Buy.Prc", "Last", "Mkt.Val", "Un.G/L",
    "Yr.High", "Yr.Low", "Day.High", "Day.Low",
    "Vol", "Lot.Size", "Chg", "Chg(Perc)",
    "Un.G/L(Perc)", "Qty.Sold", "Qty.Susp",
    "Bid", "Ask", "Avg.Sell.Price"
]

pdf = clean_numeric_columns(pdf, numeric_cols)


# Convert Pandas → Spark DataFrame
# Recommended: Sanitized column names to avoid Parquet/SparkSQL errors
schema = StructType([
    StructField("Symbol", StringType(), True),
    StructField("Code", DoubleType(), True),
    StructField("LACP", DoubleType(), True),
    StructField("Qty_Hand", IntegerType(), True),         # Replaced .
    StructField("Qty_Avai", IntegerType(), True),         # Replaced .
    StructField("Qty_Q_S", IntegerType(), True),          # Replaced .()
    StructField("Avg_Buy_Prc", DoubleType(), True),       # Replaced .
    StructField("Last", DoubleType(), True),
    StructField("Mkt_Val", DoubleType(), True),           # Replaced .
    StructField("Un_G_L", DoubleType(), True),            # Replaced ./
    StructField("Yr_High", DoubleType(), True),           # Replaced .
    StructField("Yr_Low", DoubleType(), True),            # Replaced .
    StructField("Day_High", DoubleType(), True),          # Replaced .
    StructField("Day_Low", DoubleType(), True),           # Replaced .
    StructField("Vol", LongType(), True),                 # LongType for large volume
    StructField("Lot_Size",  DoubleType(), True),         # Replaced .
    StructField("Chg",   DoubleType(), True),
    StructField("Chg_Perc", DoubleType(), True),          # Replaced ()
    StructField("Un_G_L_Perc",  StringType(), True),       # Replaced .()
    StructField("Currency", StringType(), True),
    StructField("Exchg", IntegerType(), True),
    StructField("Qty_Sold", IntegerType(), True),         # Replaced .
    StructField("Qty_Susp", IntegerType(), True),         # Replaced .
    StructField("Bid", DoubleType(), True),
    StructField("Ask", DoubleType(), True),
    StructField("Avg_Sell_Price", DoubleType(), True),     # Replaced .
    StructField("Name", StringType(), True), 
    StructField("Current_Price", DoubleType(), True) 
])

df = spark.createDataFrame(pdf, schema = schema)
display(df)
df.write.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.default.stocks")